## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
# Create DataFrame from SQL query
df = spark.sql("SELECT * FROM sales_december_2019")
display(df)


Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295665,Macbook Pro Laptop,1,1700,12/30/19 00:01,"136 Church St, New York City, NY 10001"
295666,LG Washing Machine,1,600.0,12/29/19 07:03,"562 2nd St, New York City, NY 10001"
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001"
295668,27in FHD Monitor,1,149.99,12/22/19 15:13,"410 6th St, San Francisco, CA 94016"
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301"
295670,AA Batteries (4-pack),1,3.84,12/31/19 22:58,"200 Jefferson St, New York City, NY 10001"
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001"
295673,Bose SoundSport Headphones,1,99.99,12/15/19 23:26,"718 Wilson St, Dallas, TX 75001"
295674,AAA Batteries (4-pack),4,2.99,12/28/19 11:51,"77 7th St, Dallas, TX 75001"


Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
# Your code here
from pyspark.sql.functions import col

dbutils.widgets.text("Product_name", "USB-C Charging Cable")

product_name = dbutils.widgets.get("Product_name")

df_filtered = df.filter(col("Product") == product_name)
display(df_filtered)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address
295667,USB-C Charging Cable,1,11.95,12/12/19 18:21,"277 Main St, New York City, NY 10001"
295669,USB-C Charging Cable,1,11.95,12/18/19 12:38,"43 Hill St, Atlanta, GA 30301"
295671,USB-C Charging Cable,1,11.95,12/16/19 15:10,"928 12th St, Portland, OR 97035"
295672,USB-C Charging Cable,2,11.95,12/13/19 09:29,"813 Hickory St, Dallas, TX 75001"
295675,USB-C Charging Cable,2,11.95,12/13/19 13:52,"594 1st St, San Francisco, CA 94016"
295679,USB-C Charging Cable,1,11.95,12/25/19 09:39,"902 2nd St, Dallas, TX 75001"
295681,USB-C Charging Cable,1,11.95,12/25/19 12:37,"79 Elm St, Boston, MA 02215"
295682,USB-C Charging Cable,1,11.95,12/23/19 19:25,"780 Elm St, Portland, OR 97035"
295686,USB-C Charging Cable,1,11.95,12/17/19 16:10,"573 Maple St, Portland, ME 04101"
295698,USB-C Charging Cable,2,11.95,12/13/19 14:32,"175 1st St, New York City, NY 10001"


As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
# Your code here
from pyspark.sql.functions import col, sum as _sum

df_num = (
    df
    .filter(col("Order ID") != "Order ID")
    .withColumn("Quantity Ordered", col("Quantity Ordered").cast("int"))
    .withColumn("Price Each", col("Price Each").cast("double"))
    .withColumn("Sales", col("Quantity Ordered") * col("Price Each"))
)

df_sales_by_product = (
    df_num
    .groupBy("Product")
    .agg(
        _sum("Sales").alias("Total Sales"),
        _sum("Quantity Ordered").alias("Products Sold")
    )
    .orderBy(col("Total Sales").desc())
)

display(df_sales_by_product)


Product,Total Sales,Products Sold
Macbook Pro Laptop,1094800.0,644
iPhone,635600.0,908
ThinkPad Laptop,540994.5899999965,541
Google Phone,429600.0,716
27in 4K Gaming Monitor,335781.3899999959,861
34in Ultrawide Monitor,322611.50999999605,849
Apple Airpods Headphones,311850.0,2079
Flatscreen TV,199500.0,665
Bose SoundSport Headphones,182481.74999999825,1825
27in FHD Monitor,144740.3500000011,965


In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
# Your code here
from pyspark.sql.functions import col, to_timestamp

df_typed = (
    df
    .filter(col("Order ID") != "Order ID")
    .withColumn("Order ID", col("Order ID").cast("int"))
    .withColumn("Quantity Ordered", col("Quantity Ordered").cast("int"))
    .withColumn("Price Each", col("Price Each").cast("double"))
    .withColumn("Order Date", to_timestamp(col("Order Date"), "M/d/yy H:mm"))
)

print("Row count:", df_typed.count())

display(df_typed.summary("count", "min", "max"))

Row count: 24989


summary,Order ID,Product,Quantity Ordered,Price Each,Purchase Address
count,24989,24989,24989,24989,24989
min,295665,20in Monitor,1,2.99,"1 12th St, San Francisco, CA 94016"
max,319670,iPhone,7,1700.0,"999 West St, Los Angeles, CA 90001"



5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
#your code here
from pyspark.sql.functions import lit

df_with_col = df_typed.withColumn("Data_Source", lit("Sales_December_2019"))
display(df_with_col)

Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Data_Source
295665,Macbook Pro Laptop,1,1700.0,2019-12-30T00:01:00.000Z,"136 Church St, New York City, NY 10001",Sales_December_2019
295666,LG Washing Machine,1,600.0,2019-12-29T07:03:00.000Z,"562 2nd St, New York City, NY 10001",Sales_December_2019
295667,USB-C Charging Cable,1,11.95,2019-12-12T18:21:00.000Z,"277 Main St, New York City, NY 10001",Sales_December_2019
295668,27in FHD Monitor,1,149.99,2019-12-22T15:13:00.000Z,"410 6th St, San Francisco, CA 94016",Sales_December_2019
295669,USB-C Charging Cable,1,11.95,2019-12-18T12:38:00.000Z,"43 Hill St, Atlanta, GA 30301",Sales_December_2019
295670,AA Batteries (4-pack),1,3.84,2019-12-31T22:58:00.000Z,"200 Jefferson St, New York City, NY 10001",Sales_December_2019
295671,USB-C Charging Cable,1,11.95,2019-12-16T15:10:00.000Z,"928 12th St, Portland, OR 97035",Sales_December_2019
295672,USB-C Charging Cable,2,11.95,2019-12-13T09:29:00.000Z,"813 Hickory St, Dallas, TX 75001",Sales_December_2019
295673,Bose SoundSport Headphones,1,99.99,2019-12-15T23:26:00.000Z,"718 Wilson St, Dallas, TX 75001",Sales_December_2019
295674,AAA Batteries (4-pack),4,2.99,2019-12-28T11:51:00.000Z,"77 7th St, Dallas, TX 75001",Sales_December_2019


Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
# Your code here for view creation
df_typed.createOrReplaceTempView("sales_temp_view")

In [0]:
%sql
-- Your SQL code here
SELECT Product,
       SUM(`Quantity Ordered`) AS total_units,
       ROUND(SUM(`Price Each` * `Quantity Ordered`), 2) AS total_sales
FROM sales_temp_view
GROUP BY Product
ORDER BY total_sales DESC

Product,total_units,total_sales
Macbook Pro Laptop,644,1094800.0
iPhone,908,635600.0
ThinkPad Laptop,541,540994.59
Google Phone,716,429600.0
27in 4K Gaming Monitor,861,335781.39
34in Ultrawide Monitor,849,322611.51
Apple Airpods Headphones,2079,311850.0
Flatscreen TV,665,199500.0
Bose SoundSport Headphones,1825,182481.75
27in FHD Monitor,965,144740.35
